# Random Forest Accelerator — PYNQ Z2 Deployment

This notebook deploys the **Random Forest (10 trees)** HLS accelerator on the PYNQ Z2 FPGA
and benchmarks it across all 4 datasets: Iris, Wine, Cancer, MNIST.

**Architecture:** 10 trees traversed in parallel (UNROLL on tree loop), each with pipelined
node traversal. Majority voting across all trees for final prediction.

**Interface:** AXI-Stream for data transfer (mode 0 = load forest, mode 1 = inference),
AXI-Lite for control registers.

In [ ]:
import time, struct
import numpy as np
from pynq import Overlay, allocate

AP_CTRL  = 0x00
AP_START = 0x01
AP_DONE  = 0x02

REG_MODE         = 0x10
REG_NUM_FEATURES = 0x18
REG_NUM_CLASSES  = 0x20
REG_NUM_TREES    = 0x28
REG_NUM_TEST     = 0x30
REG_LATENCY_OUT  = 0x38

def to_u32(v):
    return np.uint32(struct.unpack('<I', struct.pack('<i', int(v)))[0])

def write_reg(ip, off, val):
    ip.write(off, int(val))

def read_reg(ip, off):
    return ip.read(off)

print("Utilities loaded.")

In [ ]:
def load_rf_model(ip, dma, params):
    """Mode 0: Stream forest nodes.
    Per tree: [n_nodes] then [feat, thresh, left, right, class] x n_nodes
    """
    n_trees = int(params["n_trees"])
    tree_n_nodes = params["tree_n_nodes"].astype(np.int32)
    tree_offsets = params["tree_offsets"].astype(np.int32)
    feat   = params["feature"].astype(np.int32)
    thresh = params["threshold"].astype(np.int32)
    left   = params["children_left"].astype(np.int32)
    right  = params["children_right"].astype(np.int32)
    cls    = params["node_class"].astype(np.int32)
    n_feat = int(params["n_features"])
    n_classes = int(params["n_classes"])

    write_reg(ip, REG_MODE, 0)
    write_reg(ip, REG_NUM_FEATURES, n_feat)
    write_reg(ip, REG_NUM_CLASSES, n_classes)
    write_reg(ip, REG_NUM_TREES, n_trees)
    write_reg(ip, REG_NUM_TEST, 0)

    words = []
    for t in range(n_trees):
        nn = tree_n_nodes[t]
        words.append(to_u32(nn))
        off = tree_offsets[t]
        for i in range(nn):
            idx = off + i
            words.extend([to_u32(feat[idx]), to_u32(thresh[idx]),
                          to_u32(left[idx]), to_u32(right[idx]), to_u32(cls[idx])])

    buf_in = allocate(shape=(len(words),), dtype=np.uint32)
    buf_in[:] = np.array(words, dtype=np.uint32)
    buf_in.flush()

    write_reg(ip, AP_CTRL, AP_START)
    dma.sendchannel.transfer(buf_in)
    dma.sendchannel.wait()
    while not (read_reg(ip, AP_CTRL) & AP_DONE): pass

    buf_in.freebuffer()
    total_nodes = sum(tree_n_nodes)
    print(f"  Loaded forest: {n_trees} trees, {total_nodes} total nodes")
    return n_feat, n_classes


def run_rf_inference(ip, dma, X_test, y_true, n_feat, n_classes):
    """Mode 1: Stream test features, receive predicted classes."""
    n_test = len(y_true)
    write_reg(ip, REG_MODE, 1)
    write_reg(ip, REG_NUM_TEST, n_test)

    words = []
    for i in range(n_test):
        for j in range(n_feat):
            words.append(to_u32(X_test[i, j]))
    buf_in = allocate(shape=(len(words),), dtype=np.uint32)
    buf_out = allocate(shape=(n_test,), dtype=np.uint32)
    buf_in[:] = np.array(words, dtype=np.uint32)
    buf_in.flush()

    t0 = time.perf_counter()
    write_reg(ip, AP_CTRL, AP_START)
    dma.sendchannel.transfer(buf_in)
    dma.recvchannel.transfer(buf_out)
    dma.sendchannel.wait()
    dma.recvchannel.wait()
    while not (read_reg(ip, AP_CTRL) & AP_DONE): pass
    t1 = time.perf_counter()

    buf_out.invalidate()
    preds = np.frombuffer(buf_out, dtype=np.int32)[:n_test].copy()
    hw_cycles = read_reg(ip, REG_LATENCY_OUT)
    wall_us = (t1 - t0) * 1e6
    acc = np.sum(preds == y_true) / n_test * 100.0

    buf_in.freebuffer(); buf_out.freebuffer()
    return {"accuracy": acc, "hw_cycles": hw_cycles, "wall_us": wall_us, "preds": preds}

print("RF functions loaded.")

## Load Overlay

In [ ]:
ol = Overlay("overlays/rf_accel.bit")
ip = ol.rf_accel_0
dma = ol.axi_dma_0
print("RF overlay loaded.")
print("IP registers:", ip.register_map)

## Run on All Datasets

In [ ]:
DATASETS = ["iris", "wine", "cancer", "mnist"]
FPGA_CLK_MHZ = 100.0
results = []

for ds in DATASETS:
    print(f"\n{'='*50}")
    print(f"  RF on {ds.upper()}")
    print(f"{'='*50}")

    params = np.load(f"models/{ds}/rf_params.npz", allow_pickle=True)
    test   = np.load(f"test_vectors/{ds}/test_data.npz", allow_pickle=True)

    ol = Overlay("overlays/rf_accel.bit")
    ip = ol.rf_accel_0
    dma = ol.axi_dma_0

    n_feat, n_cls = load_rf_model(ip, dma, params)

    n_test = int(test["n_test"])
    X_test = test["X_test_gs_q"][:n_test]  # RF uses global-scaled data
    y_true = test["y_test"][:n_test]

    r = run_rf_inference(ip, dma, X_test, y_true, n_feat, n_cls)
    r["dataset"] = ds
    r["fpga_latency_us"] = r["hw_cycles"] / FPGA_CLK_MHZ
    results.append(r)

    print(f"  Accuracy:       {r['accuracy']:.2f}%")
    print(f"  HW Cycles:      {r['hw_cycles']}")
    print(f"  FPGA Latency:   {r['fpga_latency_us']:.2f} µs")
    print(f"  Wall Time:      {r['wall_us']:.1f} µs")

## Results Summary

In [ ]:
import pandas as pd

df = pd.DataFrame([{"Dataset": r["dataset"], "Accuracy (%)": r["accuracy"],
                     "HW Cycles": r["hw_cycles"],
                     "FPGA Latency (µs)": r["fpga_latency_us"],
                     "Wall Time (µs)": r["wall_us"]} for r in results])
print("\nRandom Forest Accelerator Results:")
print(df.to_string(index=False))
df.to_csv("rf_results.csv", index=False)
print("\nSaved to rf_results.csv")

## FPGA vs ESP32 Comparison

In [ ]:
esp32_rf = {"iris": None, "wine": None, "cancer": None, "mnist": None}

print(f"{'Dataset':<10} {'FPGA(µs)':>10} {'ESP32(µs)':>10} {'Speedup':>10}")
print("-" * 42)
for r in results:
    ds = r["dataset"]
    fpga_us = r["fpga_latency_us"]
    esp_us = esp32_rf.get(ds)
    speedup = f"{esp_us/fpga_us:.1f}x" if esp_us else "X"
    print(f"{ds:<10} {fpga_us:>10.2f} {str(esp_us or 'X'):>10} {speedup:>10}")